<a href="https://colab.research.google.com/github/deburky/language-models/blob/main/gpt-oss-claude-code/colab-finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Fine-tune gpt-oss-20b on Tool-Use Examples

Trains a LoRA adapter on your custom `data/train.jsonl` (pre-rendered gpt-oss harmony format).  
Requires **A100 40GB** minimum, **H100 80GB** recommended.

Tested with Colab Pro on `NVIDIA RTX PRO 6000 Blackwell Server Edition 102.0 GB`.

**Before running:** upload `data/train.jsonl` and `data/valid.jsonl` to this Colab session  
(drag & drop into the file browser, or mount Google Drive).

Inspired by: [Fine-tuning with gpt-oss and Hugging Face Transformers](https://developers.openai.com/cookbook/articles/gpt-oss/fine-tune-transfomers)

Authored by: [Denis Burakov](https://www.github.com/deburky)


## 1. Install dependencies


In [1]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128 -q
%pip install 'trl>=0.20.0' 'peft>=0.17.0' 'transformers>=4.55.0' datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 59.5 MB/s eta 0:00:00


In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA RTX PRO 6000 Blackwell Server Edition
102.0 GB


In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 0.00 GB
Reserved:  0.00 GB


## 2. Hugging Face login (needed to download gpt-oss-20b and push adapter)


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3. Load dataset

Option A — files uploaded directly to Colab (default).  
Option B — mount Google Drive and point `DATA_DIR` to your drive path.


In [ ]:
from datasets import load_dataset

DATA_DIR = "/content"  # default: files uploaded to session root

train_dataset = load_dataset(
    "json", data_files=f"{DATA_DIR}/train.jsonl", split="train"
)
eval_dataset = load_dataset("json", data_files=f"{DATA_DIR}/valid.jsonl", split="train")

print(f"Train: {len(train_dataset)} examples")
print(f"Valid: {len(eval_dataset)} examples")
print("\nSample (last 300 chars):")
print(train_dataset[0]["text"][-300:])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 284 examples
Valid: 71 examples

Sample (last 300 chars):
onald Fisher in the 1920s. It is consistent (converges to the true value with enough data) and asymptotically efficient (achieves the Cramér-Rao lower bound). Its weakness is that it can overfit with small samples and it ignores prior knowledge, which is where Bayesian estimation picks up.<|return|>


## 4. Load model


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config

MODEL_ID = "openai/gpt-oss-20b"
HUB_REPO = "deburky/gpt-oss-claude-code"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=Mxfp4Config(dequantize=True),
    use_cache=False,
    device_map="auto",
)
print("Model loaded.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# Check memory after model download
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 41.84 GB
Reserved:  54.59 GB


In [ ]:
messages = [
    {"role": "user", "content": "Who is Alan Turing?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

output_ids = model.generate(**inputs, max_new_tokens=512)
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1] :])
print(response)

<|channel|>analysis<|message|>We need to answer who Alan Turing is. Provide concise but comprehensive answer. Include his role as a mathematician, computer scientist, code breaker, key figure in development of theoretical computer science, Turing machine, Turing test, WWII code-breaking at Bletchley Park, etc. Maybe mention his persecution, death, legacy. Should we format? It's just one question. We'll respond appropriately.<|end|><|start|>assistant<|channel|>final<|message|>Alan Turing (1912 – 1954) was an English mathematician, logician, and computer scientist whose work laid the foundations for modern computing and artificial intelligence.  

**Key achievements**

| Area | Major Contributions |
|------|--------------------|
| **Theoretical computer science** | • Coined the *Turing machine* (1936), an abstract computing device that formalised the concept of algorithm and computation; it is still the basis of computational theory.<br>• Demonstrated the *universal Turing machine* (1936

## 5. Configure LoRA


In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 15,040,512 || all params: 20,929,797,696 || trainable%: 0.0719


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:212: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(f"Unsupported layer type '{type(module)}' encountered, proceed at your own risk.", UserWarning)


In [ ]:
peft_model.print_trainable_parameters()

messages = [{"role": "user", "content": "Who is Alan Turing?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(peft_model.device)

with torch.no_grad():
    output_ids = peft_model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1] :]))

trainable params: 15,040,512 || all params: 20,929,797,696 || trainable%: 0.0719
<|channel|>analysis<|message|>The user asks: "Who is Alan Turing?" Likely a brief answer: mathematician, computer science pioneer, WWII codebreaker, Turing award named after him, Turing machine, contributed to computer science, cryptanalysis. Could mention his life: born 1912, died 1954, persecuted for homosexuality. Maybe ask context? But presumably just ask for a general biographical summary. Maybe talk about his contributions to AI (Turing test), the Turing machine, the Bombe, computing. Provide details. No other constraints. Provide a concise but thorough answer. Possibly want in certain style; uncertain. Let's answer straightforward.

Given the style, should be clear. We'll mention major points. Let's elaborate: early life, education at King's College Cambridge, Turing machine, 1936; contributions to logic and computability; 1938 Turing test; WWII at Bletchley Park, cracking Enigma, etc; Bletchley Par

In [ ]:
peft_model.gradient_checkpointing_disable()
peft_model.eval()

messages = [{"role": "user", "content": "Who is Alan Turing?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(peft_model.device)

with torch.no_grad():
    output_ids = peft_model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1] :]))

<|channel|>analysis<|message|>We need to answer: Who is Alan Turing? Provide a concise explanation.

Probably: Alan Turing was an English mathematician, logician, computer scientist, etc., known for Turing machine, etc. He was a pioneer in computing, worked on codebreaking at Bletchley Park, etc. Also mention the Turing test. Mention his persecution and eventual death.

Use accessible language. Provide key contributions. Also mention he is considered father of theoretical computer science and AI.

Could mention his early life, education, major works.

Also mention his work on morphogenesis. Mention the Turing's "On Computable Numbers, with an Application to the Entscheidungsproblem" book in 1936.

He was persecuted for homosexuality and died 1954. His contributions were recognized later.

We can also mention the Turing Award.

Answer with maybe a paragraph and bullet list of key achievements.<|end|><|start|>assistant<|channel|>final<|message|>Alan Turing (1912 – 1954) was an English ma

## 6. Train and test


In [9]:
from huggingface_hub import login
import getpass

WRITE_TOKEN = getpass.getpass("HF Write Token: ")
login(token=WRITE_TOKEN)

HF Write Token: ··········


In [15]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="checkpoints",
    dataset_text_field="text",
    max_length=2048,
    learning_rate=1e-4,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/71 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/71 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998}.


Step,Training Loss,Validation Loss
25,1.506594,1.264706
50,0.532651,0.524128
75,0.414120,0.483711
100,0.453991,0.476647


TrainOutput(global_step=108, training_loss=1.0118378787129014, metrics={'train_runtime': 283.6275, 'train_samples_per_second': 3.004, 'train_steps_per_second': 0.381, 'total_flos': 3.310947250997069e+16, 'train_loss': 1.0118378787129014})

In [ ]:
peft_model.gradient_checkpointing_disable()
peft_model.eval()

messages = [{"role": "user", "content": "Who is Alan Turing?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(peft_model.device)

with torch.no_grad():
    output_ids = peft_model.generate(**inputs, max_new_tokens=256)

print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1] :]))

<|channel|>final<|message|>Alan Turing was a pioneering computer scientist and mathematician, best known for his role in breaking German Enigma codes during World II and designing the "Turing Test", a test for machine intelligence. He's considered one of the founders of computer science and artificial intelligence.<|return|>


In [ ]:
import re, json, pathlib, torch

TOOLS = {
    "role": "system",
    "content": "You are a coding assistant. Use <tool_call>{...}</tool_call> for: read(path,offset,limit), glob(pat)",
}


def tool(name, args):
    if name == "read":
        lines = pathlib.Path(args["path"]).read_text().splitlines()
        o, n = int(args.get("offset", 0)), int(args.get("limit", 20))
        return "".join(f"{o + i + 1:4}| {l}\n" for i, l in enumerate(lines[o : o + n]))
    if name == "glob":
        import glob

        return "\n".join(sorted(glob.glob(args["pat"], recursive=True))) or "none"


def chat(prompt, model, tokenizer, steps=5):
    msgs = [TOOLS, {"role": "user", "content": prompt}]
    for _ in range(steps):
        inp = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inp, max_new_tokens=256, temperature=0.3, do_sample=True
            )
        raw = tokenizer.decode(
            out[0][inp["input_ids"].shape[-1] :], skip_special_tokens=False
        )
        text = re.sub(
            r"<\|[^>]+\|>",
            "",
            raw.split("<|channel|>final<|message|>")[-1]
            if "<|channel|>final" in raw
            else raw,
        ).strip()

        calls = [
            json.loads(m.group(1))
            for m in re.finditer(r"<tool_call>(.*?)</tool_call>", raw, re.DOTALL)
        ]
        if not calls:
            print(f"ANSWER: {text}")
            return

        msgs.append({"role": "assistant", "content": text})
        for c in calls:
            r = tool(c["tool"], c.get("args", {}))
            print(f"→ {c['tool']}: {r[:200]}")
            msgs.append({"role": "user", "content": f"Tool result:\n{r}"})


chat("Read the first 10 lines of /etc/hostname", peft_model, tokenizer)
chat("Who is Alan Turing?", peft_model, tokenizer)

ANSWER: <tool_call type=read arguments={path:"/etc/hostname",offset:0,limit:10} /></assistant>
ANSWER: Alan Turing (1912–1954) was a British mathematician, logician, and computer scientist. He is widely regarded as the father of theoretical computer science and artificial intelligence. Turing's work on the Entscheidungsproblem led to the concept of the universal machine (Turing machine), a foundational model of computation. He also played a key role in breaking German Enigma codes during World War II.


## 7. Merge adapter and push full model to Hub


In [ ]:
import os, gc
from safetensors.torch import save_file
from huggingface_hub import HfApi

merged = peft_model.merge_and_unload()
merged = merged.cpu()

os.makedirs("gpt-oss-claude-code", exist_ok=True)
state_dict = merged.state_dict()

max_shard = 4 * 1024**3
shards, current, current_size = [], {}, 0
weight_map, total_size = {}, 0

for k, v in state_dict.items():
    size = v.nbytes
    if current_size + size > max_shard and current:
        shards.append(current)
        current, current_size = {}, 0
    current[k] = v.contiguous()
    current_size += size
    total_size += size
if current:
    shards.append(current)

for i, shard in enumerate(shards):
    fname = f"model-{i + 1:05d}-of-{len(shards):05d}.safetensors"
    save_file(shard, f"gpt-oss-claude-code/{fname}")
    for k in shard:
        weight_map[k] = fname
    print(f"Saved {i + 1}/{len(shards)}")

with open("gpt-oss-claude-code/model.safetensors.index.json", "w") as f:
    json.dump(
        {"metadata": {"total_size": total_size}, "weight_map": weight_map}, f, indent=2
    )

config = merged.config
if hasattr(config, "quantization_config"):
    delattr(config, "quantization_config")
config.save_pretrained("gpt-oss-claude-code")
tokenizer.save_pretrained("gpt-oss-claude-code")

del merged, state_dict
gc.collect()

api = HfApi()
api.create_repo(HUB_REPO, exist_ok=True)
api.upload_folder(
    folder_path="gpt-oss-claude-code", repo_id=HUB_REPO, repo_type="model"
)
print(f"Done — pushed to {HUB_REPO}")

Saved 1/13
Saved 2/13
Saved 3/13
Saved 4/13
Saved 5/13
Saved 6/13
Saved 7/13
Saved 8/13
Saved 9/13
Saved 10/13
Saved 11/13
Saved 12/13
Saved 13/13


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...laude-code/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...0001-of-00013.safetensors:   0%|          |  537kB / 3.92GB            

...

  ...0007-of-00013.safetensors:   3%|2         | 94.3MB / 3.29GB            

  ...0006-of-00013.safetensors:   2%|2         | 80.1MB / 3.29GB            

  ...0005-of-00013.safetensors:   7%|7         |  240MB / 3.29GB            

  ...0011-of-00013.safetensors:   7%|6         |  216MB / 3.29GB            

Done — pushed to deburky/gpt-oss-claude-code


## 8. Inference (merge adapter + base model)

> Restart kernel before running this section to free GPU memory.


In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deburky/gpt-oss-claude-code")

model = AutoModelForCausalLM.from_pretrained(
    "deburky/gpt-oss-claude-code",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Model ready.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

Model ready.


In [ ]:
messages = [{"role": "user", "content": "Who is Alan Turing?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256)

response = tokenizer.decode(out[0][inputs["input_ids"].shape[-1] :])
if "<|channel|>final<|message|>" in response:
    response = response.split("<|channel|>final<|message|>")[-1]
print(re.sub(r"<\|[^>]+\|>", "", response).strip())

Alan Mathison Turing was a pioneering British mathematician, computer scientist, and logician, best known for his critical role in cracking German Enigma codes during World War II and for laying theoretical foundations of artificial intelligence through his question "Can Machines Think?".

Key Points

 - Born in 1912, died in 1954; his early work on computability led to the formal concept of a "Turing machine," an abstract computer that underpin modern computing.
 - At Bletchley Park during WWII, he built the "Colossus," the world’s first programmable electronic computer, dramatically shortening Allied code-breaking time.
 - Post-war, he worked on early British computers and proposed early AI tests, such as "The Turing Test," where an indistinguishable human dialogue would be a sign of machine sentience.
 - Later in life, he faced intense persecution; he was chemically castrated and murdered, and it wasn't until 2009 that he was officially pardoned by the UK government for being "heter